In [ ]:
import pandas as pf
import numpy as np
import yfinance as yf
import requests
import io
import pandas as pd

In [ ]:
MACRO = {
    "^NDX": "Nasdaq-100",       # Nasdaq-100 index
    "^VIX": "Volatility",       # implied volatility (fear index)
    "^TNX": "10y_treasury_yield",       # 10-year Treasury yield
    "^IRX": "3month_treasury_yield",       # 3-month Treasury yield
    "CL=F": "crude_oil",       # WTI crude oil futures
    "DX-Y.NYB": "US_dollar",   # US dollar index
    "GC=F": "gold", 
    "SI=F": "silver",  # gold futures
}

tickers = list(MACRO.keys())
print(tickers)

In [ ]:
NASDAQ_FEATURES = {
    # --- INDICES ET VOLATILITÉ ---
    "^IXIC": "Nasdaq_composite",              # Broad tech market index
    "^VXN": "Nasdaq_volatility",              # Nasdaq-100 implied volatility (Tech fear index)
    "^VIX": "Volatility",                     # S&P 500 implied volatility (Market fear index)

    # --- interest ---
    "^IRX": "3month_treasury_yield",          # 3-month Treasury yield
    "^FVX": "5y_treasury_yield",              # 5-year Treasury yield
    "^TNX": "10y_treasury_yield",             # 10-year Treasury yield (Tech discounting rate)
    "^TYX": "30y_treasury_yield",             # 30-year Treasury yield

    # --- MATIÈRES PREMIÈRES ET DEVISES ---
    "DX-Y.NYB": "US_dollar",                  # US dollar index (DXY)
    "CL=F": "crude_oil",                      # WTI crude oil futures
    "GC=F": "gold",                           # Gold futures
    "SI=F": "silver",                         # Silver futures
    "BTC-USD": "Bitcoin",                     # Bitcoin USD (Risk-on sentiment proxy)
}

tickers1 = list(NASDAQ_FEATURES.keys())
print(tickers1)

In [ ]:
html = requests.get("https://en.wikipedia.org/wiki/List_of_NASDAQ-100_companies", headers={"User-Agent": "Mozilla/5.0"}).text
table = pd.read_html(io.StringIO(html))
ndx = next(t for t in table if "Ticker" in t.columns)
TICKERS = ndx["Ticker"].str.replace(".", "-", regex=False).tolist()

In [ ]:
TICKERS = list(dict.fromkeys(TICKERS + tickers1)) 


In [ ]:
print(TICKERS)

In [ ]:

# We download the market 
df = yf.download(TICKERS, start='2018-01-01', end='2026-10-12')[['Open', 'Close','Volume']]

df


In [ ]:
df.isnull().sum().sort_values(ascending=False)/len(df)

We observe a lot of missing values in various features. We're gonna proceed by imputating and droping columns, based on amount of missing values.

In [ ]:
# columns are (field, ticker): rename the ticker level with MACRO
df = df.rename(columns=MACRO, level=1)
df.head()

In [ ]:
open  = df["Open"]["Nasdaq-100"]
close = df["Close"]["Nasdaq-100"]


In [ ]:
df["up/down"] = (close >= open).astype(int)

In [ ]:
def date_split(df):
    d = df.index
    out = df.copy()
    out["day"] = d.day
    out["month"] = d.month
    out["day_month"] = d.dayofyear
    return out

In [ ]:
df_dates = date_split(df)

In [ ]:
df_dates

In [ ]:
df_dates.to_feather("../data/treated_dates.feather")